# Homework 6: Mạng Nơ-ron Hồi quy (RNN) để Dự đoán Tên ở Cấp độ Ký tự
## Môn học: Trí Tuệ Nhân Tạo - EE3063

Nhóm Thực Hiện: [Điền tên thành viên nhóm]
MSSV: [Điền MSSV thành viên nhóm]

Nội dung:
1. Giới thiệu về RNN và bài toán dự đoán ký tự.
2. Chuẩn bị dữ liệu tên và mã hóa one-hot.
3. Xây dựng mô hình RNN đơn giản từ đầu.
4. Huấn luyện mô hình.
5. Dự đoán ký tự và sinh tên.

In [31]:
# -*- coding: utf-8 -*-
import numpy as np

## 1. Giới thiệu

**Mạng Nơ-ron Hồi quy (Recurrent Neural Network - RNN)** là một lớp các mạng nơ-ron nhân tạo được thiết kế đặc biệt để xử lý dữ liệu dạng chuỗi (sequential data), ví dụ như văn bản, chuỗi thời gian, âm thanh. Điểm đặc biệt của RNN là nó có các kết nối vòng (recurrent connections), cho phép thông tin từ các bước thời gian trước đó được "ghi nhớ" và ảnh hưởng đến việc xử lý ở các bước thời gian hiện tại và tương lai.

Trong bài tập này, chúng ta sẽ xây dựng một mô hình RNN đơn giản ở cấp độ ký tự (character-level) để học cách dự đoán ký tự tiếp theo trong một tên, dựa trên các ký tự đã xuất hiện trước đó. Ví dụ, nếu đầu vào là "B", mô hình sẽ cố gắng dự đoán "i"; nếu đầu vào là "Bi", mô hình sẽ cố gắng dự đoán "n", v.v.

Dataset cho bài toán này là một tập hợp nhỏ các tên: "Bình", "Long", "Dũng".

### Biểu diễn Dữ liệu
Mỗi ký tự sẽ được biểu diễn dưới dạng vector **one-hot encoding**. Ví dụ, nếu bộ từ vựng của chúng ta là {'B', 'ì', 'n', 'h', 'L', 'o', 'g', 'D', 'ũ'}, thì ký tự 'B' có thể được mã hóa thành [1,0,0,0,0,0,0,0,0], 'ì' thành [0,1,0,0,0,0,0,0,0], v.v.

* Mỗi ký tự đầu vào tại mỗi bước thời gian ($x_t$) sẽ được chuyển đổi thành một **vector cột one-hot** ngay từ giai đoạn chuẩn bị dữ liệu.
* Chuỗi đầu vào cho hàm `train_sequence` sẽ là một danh sách các vector one-hot này.
* Hàm `forward_step` của RNN sẽ nhận trực tiếp vector one-hot làm đầu vào.
* Sẽ có ví dụ in ra để minh họa cách một chuỗi ký tự được chuyển thành chuỗi các vector one-hot.

### Kiến trúc RNN cơ bản
Mô hình RNN của chúng ta sẽ bao gồm:
1.  **Lớp đầu vào (Input Layer):** Nhận vector one-hot của ký tự hiện tại ($x_t$).
2.  **Lớp ẩn (Hidden Layer):** Tính toán trạng thái ẩn hiện tại ($h_t$) dựa trên $x_t$ và trạng thái ẩn trước đó ($h_{t-1}$).
    $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$
3.  **Lớp đầu ra (Output Layer):** Từ $h_t$, tính toán một vector điểm số (logits) cho mỗi ký tự trong bộ từ vựng.
    $o_t = W_{hy}h_t + b_y$
4.  **Hàm Softmax:** Chuyển đổi vector điểm số $o_t$ thành một phân phối xác suất $\hat{y}_t$ trên các ký tự, cho biết xác suất ký tự tiếp theo là gì.
    $\hat{y}_t = \text{softmax}(o_t)$

### Huấn luyện
-   **Hàm mất mát (Loss Function):** Sử dụng Cross-Entropy loss để so sánh phân phối xác suất dự đoán $\hat{y}_t$ với phân phối xác suất thực tế của ký tự tiếp theo (là một vector one-hot của ký tự đúng).

-   **Lan truyền ngược theo thời gian (Backpropagation Through Time - BPTT):** Tính toán gradient của hàm mất mát theo các tham số của mạng ($W_{xh}, W_{hh}, W_{hy}, b_h, b_y$) và cập nhật chúng bằng một thuật toán tối ưu (ví dụ: Stochastic Gradient Descent - SGD).

-  Hàm `generate_name_rnn` sẽ chỉ sử dụng ký tự đầu tiên (dưới dạng one-hot) làm input thực sự.

- Các bước sinh ký tự sau đó sẽ sử dụng một input "giả" (vector zero dạng cột) và mô hình phải dựa vào thông tin lưu trữ trong trạng thái ẩn để tiếp tục sinh ra phần còn lại của tên.

## 2. Chuẩn bị Dữ liệu và One-Hot Encoding

In [32]:
# ## 1. Giới thiệu và Mục tiêu Điều chỉnh
# 
# Dựa trên các phản hồi và yêu cầu làm rõ từ giáo viên, chúng ta thực hiện các điều chỉnh sau, đặc biệt nhấn mạnh vào kỹ thuật **One-Hot Encoding** và cách mô hình sinh output:
# 
# 1.  **Biểu diễn Input bằng One-Hot Encoding một cách tường minh:**

# 
# 2.  **Cơ chế Sinh Tên chỉ từ Ký tự Đầu:**

# Mục tiêu là đảm bảo mô hình được xây dựng và huấn luyện theo đúng kỹ thuật one-hot encoding cho input và kiểm tra khả năng "ghi nhớ" và "tự sinh" của RNN khi chỉ có thông tin ban đầu.
# """

data_rnn = ["Bình", "Long", "Dũng"]

# Tạo bộ từ vựng
chars_set = set()
for name in data_rnn:
    for char in name:
        chars_set.add(char)

sorted_chars = sorted(list(chars_set))
char_to_int = {ch: i for i, ch in enumerate(sorted_chars)}
int_to_char = {i: ch for i, ch in enumerate(sorted_chars)}
vocab_size = len(sorted_chars)

print(f"Bộ từ vựng RNN ({vocab_size} ký tự): {sorted_chars}")
print(f"Ánh xạ ký tự sang số (dùng để lấy index cho one-hot): {char_to_int}")

def to_one_hot_column_vector(char_idx, vocab_size):
    """Chuyển chỉ số ký tự thành vector one-hot dạng cột."""
    vec = np.zeros((vocab_size, 1)) 
    vec[char_idx, 0] = 1            
    return vec

# Chuẩn bị dữ liệu huấn luyện với input là chuỗi các vector one-hot (dạng cột)
# và target là chuỗi các chỉ số nguyên
training_data_rnn_final = []
for name_str in data_rnn:
    if len(name_str) > 1:
        input_one_hot_sequence = []
        target_indices_sequence = []
        for i in range(len(name_str) - 1):
            input_char = name_str[i]
            target_char = name_str[i+1]
            
            input_one_hot_sequence.append(to_one_hot_column_vector(char_to_int[input_char], vocab_size))
            target_indices_sequence.append(char_to_int[target_char])
        
        if input_one_hot_sequence:
            training_data_rnn_final.append((input_one_hot_sequence, target_indices_sequence))

# --- In ví dụ One-Hot Encoding ---
print("\n--- Ví dụ One-Hot Encoding cho tên đầu tiên trong dữ liệu huấn luyện ---")
if training_data_rnn_final:
    sample_name_str = data_rnn[0]
    sample_input_one_hot_seq, sample_target_indices_seq = training_data_rnn_final[0]
    print(f"Tên mẫu: '{sample_name_str}'")
    print("Chuỗi input (one-hot) và target (index) tương ứng:")
    for t in range(len(sample_input_one_hot_seq)):
        input_char = sample_name_str[t]
        target_char = sample_name_str[t+1]
        print(f"  Bước {t}:")
        print(f"    Input ký tự: '{input_char}' -> Vector One-Hot (dạng cột, {vocab_size}x1):\n{sample_input_one_hot_seq[t].T}") # .T để in dễ nhìn hơn
        print(f"    Target ký tự: '{target_char}' -> Index: {sample_target_indices_seq[t]}")
else:
    print("Không có dữ liệu huấn luyện để hiển thị ví dụ one-hot.")


Bộ từ vựng RNN (9 ký tự): ['B', 'D', 'L', 'g', 'h', 'n', 'o', 'ì', 'ũ']
Ánh xạ ký tự sang số (dùng để lấy index cho one-hot): {'B': 0, 'D': 1, 'L': 2, 'g': 3, 'h': 4, 'n': 5, 'o': 6, 'ì': 7, 'ũ': 8}

--- Ví dụ One-Hot Encoding cho tên đầu tiên trong dữ liệu huấn luyện ---
Tên mẫu: 'Bình'
Chuỗi input (one-hot) và target (index) tương ứng:
  Bước 0:
    Input ký tự: 'B' -> Vector One-Hot (dạng cột, 9x1):
[[1. 0. 0. 0. 0. 0. 0. 0. 0.]]
    Target ký tự: 'ì' -> Index: 7
  Bước 1:
    Input ký tự: 'ì' -> Vector One-Hot (dạng cột, 9x1):
[[0. 0. 0. 0. 0. 0. 0. 1. 0.]]
    Target ký tự: 'n' -> Index: 5
  Bước 2:
    Input ký tự: 'n' -> Vector One-Hot (dạng cột, 9x1):
[[0. 0. 0. 0. 0. 1. 0. 0. 0.]]
    Target ký tự: 'h' -> Index: 4


## 3. Xây dựng Mô hình RNN

In [33]:
def tanh_rnn(x):
    return np.tanh(x)

def softmax_rnn(x):
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return e_x / np.sum(e_x, axis=0, keepdims=True)

class SimpleRNNFinalAdjusted:
    def __init__(self, vocab_size, hidden_size, learning_rate=0.01):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = learning_rate
        # W_xh: input (one-hot) to hidden
        self.W_xh = np.random.randn(hidden_size, vocab_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hy = np.random.randn(vocab_size, hidden_size) * 0.01
        self.b_h = np.zeros((hidden_size, 1))
        self.b_y = np.zeros((vocab_size, 1))

    def forward_step(self, x_one_hot_input, h_prev):
        """
        Thực hiện một bước lan truyền xuôi.
        Args:
            x_one_hot_input (np.array): Vector one-hot của ký tự đầu vào, shape (vocab_size, 1).
            h_prev (np.array): Trạng thái ẩn từ bước thời gian trước.
        Returns:
            y_pred_proba, h_current, o_logits
        """
        # h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)
        h_current = tanh_rnn(np.dot(self.W_xh, x_one_hot_input) + np.dot(self.W_hh, h_prev) + self.b_h)
        # o_t = W_hy * h_t + b_y
        o_logits = np.dot(self.W_hy, h_current) + self.b_y
        y_pred_proba = softmax_rnn(o_logits)
        return y_pred_proba, h_current, o_logits

    def train_sequence(self, one_hot_input_seq, target_indices_seq):
        loss = 0
        h_prev = np.zeros((self.hidden_size, 1))
        # Lưu trữ các giá trị trung gian cho BPTT
        # xs_one_hot đã là đầu vào, hs (hidden states), os_logits, ys_pred_probas
        hs_cache, os_logits_cache, ys_pred_probas_cache = {}, {}, {}
        hs_cache[-1] = np.copy(h_prev)

        # --- Lan truyền xuôi ---
        for t in range(len(one_hot_input_seq)):
            x_one_hot_t = one_hot_input_seq[t] # Input đã là one-hot
            y_target_idx_t = target_indices_seq[t]
            
            y_pred_proba_t, h_current_t, o_logits_t = self.forward_step(x_one_hot_t, hs_cache[t-1])
            
            hs_cache[t] = h_current_t
            os_logits_cache[t] = o_logits_t
            ys_pred_probas_cache[t] = y_pred_proba_t
            
            loss_t = -np.log(y_pred_proba_t[y_target_idx_t, 0] + 1e-9) # Thêm epsilon để tránh log(0)
            loss += loss_t
        
        # --- Lan truyền ngược theo thời gian (BPTT) ---
        dW_xh, dW_hh, dW_hy = np.zeros_like(self.W_xh), np.zeros_like(self.W_hh), np.zeros_like(self.W_hy)
        db_h, db_y = np.zeros_like(self.b_h), np.zeros_like(self.b_y)
        dh_next_bptt = np.zeros_like(hs_cache[0]) # Gradient của loss theo h_t, truyền từ bước t+1 về t

        for t in reversed(range(len(one_hot_input_seq))):
            y_target_idx_t = target_indices_seq[t]
            
            # Gradient của loss theo output logits o_t (dL/do_t)
            dy_logits = np.copy(ys_pred_probas_cache[t])
            dy_logits[y_target_idx_t] -= 1 
            
            # Gradient cho W_hy và b_y
            dW_hy += np.dot(dy_logits, hs_cache[t].T)
            db_y += dy_logits
            
            # Gradient của loss theo hidden state h_t (dL/dh_t)
            dh_t = np.dot(self.W_hy.T, dy_logits) + dh_next_bptt
            
            # Gradient qua hàm tanh: dtanh = (1 - h_t^2)
            dh_raw_input_to_tanh = (1 - hs_cache[t]**2) * dh_t
            
            # Gradient cho b_h
            db_h += dh_raw_input_to_tanh
            
            # Gradient cho W_xh
            # Input cho bước này là one_hot_input_seq[t]
            dW_xh += np.dot(dh_raw_input_to_tanh, one_hot_input_seq[t].T)
            
            # Gradient cho W_hh
            dW_hh += np.dot(dh_raw_input_to_tanh, hs_cache[t-1].T)
            
            # Cập nhật dh_next_bptt cho bước lặp ngược tiếp theo
            dh_next_bptt = np.dot(self.W_hh.T, dh_raw_input_to_tanh)

        # Giới hạn gradient (gradient clipping)
        for dparam in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
            np.clip(dparam, -5, 5, out=dparam) 
            
        avg_loss = loss / len(one_hot_input_seq)
        gradients = {'dW_xh': dW_xh, 'dW_hh': dW_hh, 'dW_hy': dW_hy, 'db_h': db_h, 'db_y': db_y}
        
        return avg_loss, gradients

    def update_parameters(self, gradients):
        self.W_xh -= self.lr * gradients['dW_xh']
        self.W_hh -= self.lr * gradients['dW_hh']
        self.W_hy -= self.lr * gradients['dW_hy']
        self.b_h  -= self.lr * gradients['db_h']
        self.b_y  -= self.lr * gradients['db_y']

## 4. Huấn luyện Mô hình RNN

Chúng ta sẽ huấn luyện mô hình trên các chuỗi ký tự từ bộ dữ liệu tên.

In [34]:
hidden_size_rnn = 25 
learning_rate_rnn = 0.01
n_epochs_rnn = 1500 

rnn_model = SimpleRNNFinalAdjusted(vocab_size, hidden_size_rnn, learning_rate_rnn)

print(f"\nBắt đầu huấn luyện RNN (Final Adjusted - Explicit OneHot) với hidden_size={hidden_size_rnn}, lr={learning_rate_rnn}...")
for epoch in range(n_epochs_rnn):
    total_loss_epoch = 0
    for one_hot_input_seq, target_indices_seq in training_data_rnn_final:
        loss, grads = rnn_model.train_sequence(one_hot_input_seq, target_indices_seq)
        rnn_model.update_parameters(grads)
        total_loss_epoch += loss
    avg_loss_epoch = total_loss_epoch / len(training_data_rnn_final) if training_data_rnn_final else 0
    if (epoch + 1) % 300 == 0:
        print(f"Epoch {epoch+1}/{n_epochs_rnn}, Loss trung bình: {avg_loss_epoch:.4f}")
print("Hoàn tất huấn luyện RNN (Final Adjusted - Explicit OneHot).")


Bắt đầu huấn luyện RNN (Final Adjusted - Explicit OneHot) với hidden_size=25, lr=0.01...
Epoch 300/1500, Loss trung bình: 0.6364
Epoch 600/1500, Loss trung bình: 0.1662
Epoch 900/1500, Loss trung bình: 0.0175
Epoch 1200/1500, Loss trung bình: 0.0090
Epoch 1500/1500, Loss trung bình: 0.0060
Hoàn tất huấn luyện RNN (Final Adjusted - Explicit OneHot).


## 5. Dự đoán Ký tự và Sinh Tên

Sau khi huấn luyện, chúng ta có thể sử dụng mô hình để:
1.  Dự đoán ký tự tiếp theo cho một chuỗi ký tự đầu vào.
2.  Sinh ra một chuỗi tên mới bằng cách lấy mẫu từ phân phối xác suất đầu ra.

In [35]:
def generate_name_rnn(model, start_char_str, max_len_name):
    if start_char_str not in char_to_int:
        print(f"Ký tự '{start_char_str}' không có trong từ vựng.")
        return start_char_str
        
    generated_name = start_char_str
    h_col_vec = np.zeros((model.hidden_size, 1))
    
    start_char_idx = char_to_int[start_char_str]
    x_t_one_hot_col_vec = to_one_hot_column_vector(start_char_idx, model.vocab_size)
    
    y_pred_proba, h_next_col_vec, _ = model.forward_step(x_t_one_hot_col_vec, h_col_vec)
    h_col_vec = h_next_col_vec
    
    predicted_idx = np.argmax(y_pred_proba.flatten())
    generated_name += int_to_char[predicted_idx]
    
    dummy_input_one_hot_col_vec = np.zeros((model.vocab_size, 1))
    
    for _ in range(max_len_name - len(start_char_str) - 1): 
        if len(generated_name) >= max_len_name: break
        y_pred_proba, h_next_col_vec, _ = model.forward_step(dummy_input_one_hot_col_vec, h_col_vec)
        h_col_vec = h_next_col_vec
        predicted_idx = np.argmax(y_pred_proba.flatten())
        generated_name += int_to_char[predicted_idx]
    return generated_name

print("\n--- Sinh Tên theo Yêu cầu Mới (RNN Final Adjusted - Explicit OneHot) ---")
target_names = ["Bình", "Long", "Dũng"]
start_chars = ["B", "L", "D"]

for i, start_char in enumerate(start_chars):
    desired_length = len(target_names[i]) 
    generated_name_result = generate_name_rnn(rnn_model, start_char, desired_length)
    print(f"Bắt đầu bằng '{start_char}', Hoàn thành: '{generated_name_result}' (Mong muốn: '{target_names[i]}')")

   


--- Sinh Tên theo Yêu cầu Mới (RNN Final Adjusted - Explicit OneHot) ---
Bắt đầu bằng 'B', Hoàn thành: 'Bình' (Mong muốn: 'Bình')
Bắt đầu bằng 'L', Hoàn thành: 'Long' (Mong muốn: 'Long')
Bắt đầu bằng 'D', Hoàn thành: 'Dũng' (Mong muốn: 'Dũng')


## 6. Nhận xét và Kết luận

-   Mô hình RNN đơn giản ở cấp độ ký tự đã được xây dựng từ đầu, bao gồm các bước lan truyền xuôi, tính toán hàm mất mát (cross-entropy), và lan truyền ngược theo thời gian (BPTT) để cập nhật trọng số.
-   Mô hình được huấn luyện trên một tập dữ liệu nhỏ gồm ba tên ("Bình", "Long", "Dũng").
-   **Kết quả dự đoán:**
    * Với tập dữ liệu rất nhỏ này và mô hình RNN đơn giản, khả năng dự đoán chính xác hoàn toàn các tên có thể bị hạn chế. Mô hình có thể học được một số quy luật cơ bản về các cặp ký tự thường xuất hiện (ví dụ: sau 'B' thường là 'ì').
    * Giá trị `hidden_size`, `learning_rate`, và số `epochs` ảnh hưởng lớn đến kết quả. Cần tinh chỉnh các tham số này để có kết quả tốt hơn.
    * Việc khởi tạo trọng số ngẫu nhiên cũng có thể dẫn đến các kết quả khác nhau giữa các lần chạy.
-   **Hạn chế của mô hình RNN cơ bản:**
    * **Vanishing/Exploding Gradients:** Với các chuỗi dài, RNN cơ bản gặp khó khăn trong việc học các phụ thuộc xa do vấn đề gradient biến mất hoặc bùng nổ. Bài toán này với các tên ngắn có thể ít bị ảnh hưởng nghiêm trọng.
    * **Khả năng nhớ ngắn hạn:** Bộ nhớ của RNN cơ bản có giới hạn.
-   **Hướng cải thiện (cho các bài toán phức tạp hơn):**
    * Sử dụng các kiến trúc RNN phức tạp hơn như LSTM hoặc GRU (sẽ được đề cập trong HW7).
    * Tăng kích thước tập dữ liệu huấn luyện.
    * Tinh chỉnh các siêu tham số (hyperparameters).
    * Sử dụng các kỹ thuật regularization (ví dụ: dropout) nếu mô hình bị overfitting (khó xảy ra với tập dữ liệu nhỏ này).

 -   **One-Hot Encoding Tường Minh:**
    * Dữ liệu đầu vào cho hàm `train_sequence` giờ đây là một chuỗi các vector one-hot dạng cột.
    * Hàm `forward_step` cũng nhận và xử lý trực tiếp các vector one-hot này.
-   **Sinh Tên từ Ký tự Đầu (với Dummy Input):**
    * Logic của hàm `generate_name_rnn_final_adj` vẫn giữ nguyên như lần điều chỉnh trước, đảm bảo sau ký tự đầu tiên, các input tiếp theo là vector zero.
-   **Kết quả và Thách thức:**
    * Với dataset cực kỳ nhỏ và yêu cầu sinh tên chỉ từ trạng thái ẩn (sau ký tự đầu, với input giả), việc mô hình học được để sinh ra chính xác các tên mục tiêu vẫn là một thách thức rất lớn.
    * Việc in ra các vector one-hot giúp làm rõ cách biểu diễn dữ liệu, nhưng không thay đổi bản chất khó khăn của bài toán với dữ liệu hạn chế.